In [97]:
import sys, os, time, warnings, glob
warnings.filterwarnings("ignore")
 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cv2
from PIL import Image
from pathlib import Path
 
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2, ResNet50
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.applications.resnet50   import preprocess_input as resnet_preprocess
from tensorflow.keras.optimizers import Adam
 
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

In [101]:
TRAIN_DIR = "//kaggle/input/datasets/meowmeowmeowmeowmeow/gtsrb-german-traffic-sign/Train"
TEST_DIR  = "/kaggle/input/datasets/meowmeowmeowmeowmeow/gtsrb-german-traffic-sign/Test"
TEST_CSV  = "/kaggle/input/datasets/meowmeowmeowmeowmeow/gtsrb-german-traffic-sign/Test.csv"

In [98]:
IMG_SIZE    = (64, 64)
BATCH_SIZE  = 32
NUM_CLASSES = 43
EPOCHS_CNN  = 30
EPOCHS_TL   = 15
SEED        = 42
 
CLASS_NAMES = {
    0:  "Speed limit (20km/h)",       1:  "Speed limit (30km/h)",
    2:  "Speed limit (50km/h)",       3:  "Speed limit (60km/h)",
    4:  "Speed limit (70km/h)",       5:  "Speed limit (80km/h)",
    6:  "End of speed limit (80km/h)",7:  "Speed limit (100km/h)",
    8:  "Speed limit (120km/h)",      9:  "No passing",
    10: "No passing veh > 3.5t",      11: "Right-of-way at intersection",
    12: "Priority road",              13: "Yield",
    14: "Stop",                       15: "No vehicles",
    16: "Veh > 3.5t prohibited",      17: "No entry",
    18: "General caution",            19: "Dangerous curve left",
    20: "Dangerous curve right",      21: "Double curve",
    22: "Bumpy road",                 23: "Slippery road",
    24: "Road narrows on right",      25: "Road work",
    26: "Traffic signals",            27: "Pedestrians",
    28: "Children crossing",          29: "Bicycles crossing",
    30: "Beware of ice/snow",         31: "Wild animals crossing",
    32: "End speed + passing limits", 33: "Turn right ahead",
    34: "Turn left ahead",            35: "Ahead only",
    36: "Go straight or right",       37: "Go straight or left",
    38: "Keep right",                 39: "Keep left",
    40: "Roundabout mandatory",       41: "End of no passing",
    42: "End no passing veh > 3.5t"
}
 

In [ ]:
base_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)
 
train_gen = base_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training', seed=SEED, shuffle=True
)
val_gen = base_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation', seed=SEED, shuffle=False
)
 
print(f"\nTraining samples   : {train_gen.samples}")
print(f"Validation samples : {val_gen.samples}")
print(f"Classes found      : {train_gen.num_classes}")

In [ ]:
class_counts = pd.Series(train_gen.classes).value_counts().sort_index()
class_labels  = [CLASS_NAMES[i] for i in class_counts.index]
 
fig, ax = plt.subplots(figsize=(18, 6))
colors  = plt.cm.RdYlGn(np.linspace(0.2, 0.9, NUM_CLASSES))
ax.bar(range(NUM_CLASSES), class_counts.values, color=colors)
ax.axhline(class_counts.mean(), color='red', linestyle='--', linewidth=1.5,
           label=f'Mean = {class_counts.mean():.0f}')
ax.set_title("GTSRB Class Distribution — Training Set", fontsize=14, fontweight='bold')
ax.set_xlabel("Class Index", fontsize=12)
ax.set_ylabel("Number of Training Images", fontsize=12)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig("eda_class_distribution.png", dpi=150)
plt.show()
 

print("  CLASS IMBALANCE SUMMARY")

print(f"  Most over-represented  : {class_counts.idxmax()} → {CLASS_NAMES[class_counts.idxmax()]}")
print(f"  Count                  : {class_counts.max()} images")
print(f"  Most under-represented : {class_counts.idxmin()} → {CLASS_NAMES[class_counts.idxmin()]}")
print(f"  Count                  : {class_counts.min()} images")
print(f"  Imbalance ratio        : {class_counts.max()/class_counts.min():.1f}×")

print("  Impact: Model will bias toward high-frequency speed-limit signs.")
print("  Mitigation: class_weight computed and passed to model.fit().\n")

In [ ]:
np.random.seed(SEED)
sampled_classes = np.random.choice(NUM_CLASSES, 9, replace=False)
 
fig, axes = plt.subplots(3, 3, figsize=(11, 11))
for ax, cls_id in zip(axes.flat, sampled_classes):
    folder = Path(TRAIN_DIR) / str(cls_id)
    imgs   = list(folder.glob("*.png"))
    img    = cv2.cvtColor(cv2.imread(str(imgs[0])), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(f"[{cls_id}] {CLASS_NAMES[cls_id][:22]}", fontsize=8, fontweight='bold')
    ax.axis('off')
plt.suptitle(
    "Sample Training Images per Class\n"
    "(Note real-world challenges: blur, low-light, partial occlusion, varying size)",
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig("eda_sample_grid.png", dpi=150)
plt.show()
 

In [ ]:
all_pixels = []
for cls_id in range(0, NUM_CLASSES, 5):
    folder = Path(TRAIN_DIR) / str(cls_id)
    for img_path in list(folder.glob("*.png"))[:10]:
        img = cv2.imread(str(img_path)).astype(np.float32) / 255.
        all_pixels.append(img.flatten())
all_pixels = np.concatenate(all_pixels)
 
plt.figure(figsize=(8, 4))
plt.hist(all_pixels, bins=50, color='steelblue', edgecolor='black', alpha=0.85)
plt.title("Pixel Intensity Distribution (normalised [0, 1])", fontsize=13)
plt.xlabel("Pixel Value", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.tight_layout()
plt.savefig("eda_pixel_histogram.png", dpi=150)
plt.show()

In [ ]:
sizes = []
for cls_id in range(0, NUM_CLASSES, 5):
    folder = Path(TRAIN_DIR) / str(cls_id)
    for p in list(folder.glob("*.png"))[:5]:
        h, w, _ = cv2.imread(str(p)).shape
        sizes.append((h, w))
heights, widths = zip(*sizes)
print(f"Raw image size range: H=[{min(heights)}–{max(heights)}px]  W=[{min(widths)}–{max(widths)}px]")
print("All will be resized to 64×64 during preprocessing.\n")

In [ ]:
aug_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=15,            
    width_shift_range=0.10,       
    height_shift_range=0.10,      
    zoom_range=0.10,              
    brightness_range=[0.5, 1.5],  
    horizontal_flip=False,        
    fill_mode='nearest'
)
 
aug_train_gen = aug_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training', seed=SEED, shuffle=True
)
aug_val_gen = aug_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation', seed=SEED, shuffle=False
)
 

In [ ]:
sample_folder = Path(TRAIN_DIR) / "14"                
sample_path   = list(sample_folder.glob("*.png"))[0]
src_img       = cv2.cvtColor(cv2.imread(str(sample_path)), cv2.COLOR_BGR2RGB)
src_arr       = src_img.astype(np.float32) / 255.
 
def apply_aug(img_arr, **params):
    """Apply a single augmentation type for display."""
    gen = ImageDataGenerator(**params)
    batch = gen.flow(np.expand_dims(img_arr, 0), batch_size=1, seed=SEED)
    return next(batch)[0]
 
augmented_variants = [
    ("Original",           src_arr),
    ("Rotated (±15°)",     apply_aug(src_arr, rotation_range=15)),
    ("Shifted (±10%)",     apply_aug(src_arr, width_shift_range=0.10, height_shift_range=0.10)),
    ("Zoomed (±10%)",      apply_aug(src_arr, zoom_range=0.10)),
    ("Brightness [0.5–1.5]", apply_aug(src_arr, brightness_range=[0.5,1.5])),
]
 
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, (title, img) in zip(axes, augmented_variants):
    ax.imshow(np.clip(img, 0, 1))
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.axis('off')
plt.suptitle(
    "Augmentation Visualisation — Class 14: Stop Sign\n"
    "Horizontal flip disabled (would change sign semantics for directional signs)",
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig("augmentation_visualisation.png", dpi=150)
plt.show()
 

In [ ]:
y_train      = aug_train_gen.classes
cw_vals      = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight = dict(enumerate(cw_vals))
print(f"Class weights — max={max(cw_vals):.2f}, min={min(cw_vals):.2f}")
print(" Passed to every model.fit() call to mitigate class imbalance.\n")

In [116]:
def detect_and_crop_sign(image_input, min_area=400):
 

    if isinstance(image_input, str):
        img = cv2.imread(image_input)
        if img is None:
            raise FileNotFoundError(f"Cannot read image: {image_input}")
    else:
        img = image_input.copy()
 
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
 
    # Red — two hue ranges (HSV wraps at 180°)
    m_r1 = cv2.inRange(hsv, np.array([0,   70,  50]),  np.array([10,  255, 255]))
    m_r2 = cv2.inRange(hsv, np.array([160, 70,  50]),  np.array([180, 255, 255]))
    # Blue informational signs
    m_b  = cv2.inRange(hsv, np.array([100, 80,  50]),  np.array([130, 255, 255]))
    # Yellow/orange warning triangles
    m_y  = cv2.inRange(hsv, np.array([15,  80,  80]),  np.array([35,  255, 255]))
 
    mask = m_r1 | m_r2 | m_b | m_y
 
    # Morphological operations — fill gaps and remove small noise
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask   = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask   = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel)
 
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
 
    # Filter: minimum area + aspect ratio close to 1 (signs are roughly square/circular)
    valid = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < min_area:
            continue
        x, y, w, h = cv2.boundingRect(cnt)
        ratio = w / (h + 1e-6)
        if 0.4 < ratio < 2.5:
            valid.append((area, x, y, w, h))
 
    if not valid:
        return None
 
    valid.sort(reverse=True)          # largest area first
    _, x, y, w, h = valid[0]
 
    # 10% padding around the bounding box
    pad = int(0.10 * max(w, h))
    x1  = max(0, x - pad)
    y1  = max(0, y - pad)
    x2  = min(img.shape[1], x + w + pad)
    y2  = min(img.shape[0], y + h + pad)
 
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    cropped = cv2.resize(img_rgb[y1:y2, x1:x2], (64, 64))
 
    return img, (x1, y1, x2 - x1, y2 - y1), cropped

In [117]:
def create_synthetic_scene(class_id, scene_size=(400, 600)):
    """
    Embeds one GTSRB crop into a grey road-like background to simulate
    a full camera frame when real street-view images are unavailable.
    Source: GTSRB training images embedded into a synthetic background.
    """
    folder  = Path(TRAIN_DIR) / str(class_id)
    img_src = cv2.imread(str(list(folder.glob("*.png"))[0]))
 
    scene = np.ones((*scene_size, 3), dtype=np.uint8) * 100  # grey road
    # Simple horizon / sky gradient
    scene[:scene_size[0]//2] = [170, 180, 200]
 
    sign_h, sign_w = 80, 80
    sign  = cv2.resize(img_src, (sign_w, sign_h))
    sy    = scene_size[0] // 3
    sx    = scene_size[1] // 2 - sign_w // 2
    scene[sy:sy+sign_h, sx:sx+sign_w] = sign
 
    return scene, (sx, sy, sign_w, sign_h)

In [ ]:
test_classes = [14, 17, 13, 25, 1]   # Stop, No Entry, Yield, Road Work, 30km/h
 
fig, axes = plt.subplots(5, 3, figsize=(15, 22))
fig.suptitle("Sign Detection & Cropping — 5 Full Road Scene Tests",
             fontsize=14, fontweight='bold')
 
for row, cls_id in enumerate(test_classes):
    scene_bgr, gt_box = create_synthetic_scene(cls_id)
    result            = detect_and_crop_sign(scene_bgr)
 
    # Column 0 — original scene
    axes[row, 0].imshow(cv2.cvtColor(scene_bgr, cv2.COLOR_BGR2RGB))
    axes[row, 0].set_title(f"Original Scene\nClass: {CLASS_NAMES[cls_id]}", fontsize=9)
    axes[row, 0].axis('off')
 
    if result is not None:
        _, (bx, by, bw, bh), cropped = result
 
        # Column 1 — bounding box overlay
        annotated = cv2.cvtColor(scene_bgr.copy(), cv2.COLOR_BGR2RGB)
        cv2.rectangle(annotated, (bx, by), (bx+bw, by+bh), (0, 255, 0), 3)
        cv2.putText(annotated, "DETECTED", (bx, max(by-5, 12)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        axes[row, 1].imshow(annotated)
        axes[row, 1].set_title("Bounding Box Overlay", fontsize=9)
 
        # Column 2 — cropped 64×64 output
        axes[row, 2].imshow(cropped)
        axes[row, 2].set_title("Cropped Sign (64×64)\n→ Ready for Classifier", fontsize=9)
    else:
        axes[row, 1].text(0.5, 0.5, "Detection Failed", ha='center',
                          va='center', color='red', transform=axes[row, 1].transAxes)
        axes[row, 1].axis('off')
        axes[row, 2].axis('off')
 
    for col in range(3):
        axes[row, col].axis('off')
 
plt.tight_layout()
plt.savefig("detection_pipeline_5scenes.png", dpi=150)
plt.show()
 
print("\nDetection limitation notes:")
print("  Colour thresholding may miss blue informational signs in high-saturation backgrounds.")
print("  Yellow warning signs can be confused with bright sunlight patches.")
print("  Heavily occluded signs fall below the minimum area threshold.")
print("  Production ADAS would use YOLO/SSD — colour thresholding is a valid baseline here.\n")

In [120]:
def build_cnn(num_hidden_layers, neurons, dropout_rate=0.3,
              input_shape=(64, 64, 3), num_classes=43):
    """
    Configurable CNN with shared convolutional backbone.
    Backbone: Conv32 → BN → Pool → Conv64 → BN → Pool → Flatten
    Head    : N × Dense(neurons, ReLU) + Dropout → Dense(43, softmax)
    BatchNormalization added for faster convergence and better regularisation.
    """
    model = models.Sequential(name=f"CNN_{num_hidden_layers}L_{neurons}N")
 
    # Convolutional backbone (fixed across all configs)
    model.add(layers.Conv2D(32, (3, 3), activation='relu',
                            padding='same', input_shape=input_shape))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))
 
    model.add(layers.Conv2D(64, (3, 3), activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))
 
    model.add(layers.Flatten())
 
    # Configurable dense head
    for _ in range(num_hidden_layers):
        model.add(layers.Dense(neurons, activation='relu'))
        model.add(layers.Dropout(dropout_rate))
 
    model.add(layers.Dense(num_classes, activation='softmax'))
 
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

In [ ]:
configs = [
    {'name': 'Config_1_Shallow', 'layers': 1, 'neurons': 128, 'dropout': 0.3},
    {'name': 'Config_2_Wide',    'layers': 1, 'neurons': 256, 'dropout': 0.4},
    {'name': 'Config_3_Deep',    'layers': 2, 'neurons': 128, 'dropout': 0.5},
]
 
cnn_results  = []
cnn_histories = []
best_cnn_val  = 0.0
best_cnn_path = ""
best_cnn_model = None
 
for cfg in configs:
    
    print(f"  Training {cfg['name']}")
    
    m = build_cnn(cfg['layers'], cfg['neurons'], cfg['dropout'])
    m.summary()
 
    ckpt_path = f"{cfg['name']}_best.keras"
    cbs = [
        callbacks.EarlyStopping(monitor='val_loss', patience=5,
                                restore_best_weights=True, verbose=1),
        callbacks.ModelCheckpoint(ckpt_path, monitor='val_accuracy',
                                  save_best_only=True, verbose=0),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                    patience=3, min_lr=1e-6, verbose=1),
    ]
 
    h = m.fit(
        aug_train_gen,
        validation_data=aug_val_gen,
        epochs=EPOCHS_CNN,
        class_weight=class_weight,
        callbacks=cbs,
        verbose=1
    )
    cnn_histories.append(h)
 
    bva = max(h.history['val_accuracy'])
    bvl = min(h.history['val_loss'])
    cnn_results.append({
        'Model Configuration': cfg['name'],
        'Hidden Layers':       cfg['layers'],
        'Neurons':             cfg['neurons'],
        'Dropout':             cfg['dropout'],
        'Best Val Accuracy (%)': round(bva * 100, 2),
        'Best Val Loss':         round(bvl, 4),
    })
 
    if bva > best_cnn_val:
        best_cnn_val   = bva
        best_cnn_path  = ckpt_path
        best_cnn_model = m
 

In [ ]:
df_cnn = pd.DataFrame(cnn_results)
print("\n CNN Configuration Comparison Table:")
print(df_cnn.to_string(index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(df_cnn['Model Configuration'],
              df_cnn['Best Val Accuracy (%)'],
              color=['#2196F3', '#4CAF50', '#FF9800'], width=0.5)
for b, v in zip(bars, df_cnn['Best Val Accuracy (%)']):
    ax.text(b.get_x() + b.get_width()/2, v + 0.3, f"{v}%",
            ha='center', fontweight='bold', fontsize=11)
ax.set_ylabel("Best Validation Accuracy (%)", fontsize=12)
ax.set_title("CNN Configurations — Validation Accuracy Comparison",
             fontsize=13, fontweight='bold')
ax.set_ylim(0, 105)
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig("cnn_accuracy_comparison.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
for h, cfg, color in zip(cnn_histories, configs, colors):
    axes[0].plot(h.history['val_accuracy'], label=cfg['name'], color=color, linewidth=1.8)
    axes[1].plot(h.history['val_loss'],     label=cfg['name'], color=color, linewidth=1.8)
for ax, title, ylabel in zip(
    axes,
    ["Validation Accuracy per Epoch", "Validation Loss per Epoch"],
    ["Accuracy", "Loss"]
):
    ax.set_xlabel("Epoch", fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)
plt.suptitle("CNN Config Training Curves", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("cnn_training_curves.png", dpi=150)
plt.show()
 
print(f"\n Best CNN : {best_cnn_path}  (Val Acc = {best_cnn_val*100:.2f}%)")

In [ ]:
mn_train_gen = ImageDataGenerator(
    preprocessing_function=mobilenet_preprocess,
    rotation_range=15, width_shift_range=0.1,
    height_shift_range=0.1, zoom_range=0.1,
    brightness_range=[0.8, 1.2], validation_split=0.2
).flow_from_directory(TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
                      class_mode='categorical', subset='training', seed=SEED)
 
mn_val_gen = ImageDataGenerator(
    preprocessing_function=mobilenet_preprocess, validation_split=0.2
).flow_from_directory(TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
                      class_mode='categorical', subset='validation', seed=SEED, shuffle=False)
 
rn_train_gen = ImageDataGenerator(
    preprocessing_function=resnet_preprocess,
    rotation_range=15, width_shift_range=0.1,
    height_shift_range=0.1, zoom_range=0.1,
    brightness_range=[0.8, 1.2], validation_split=0.2
).flow_from_directory(TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
                      class_mode='categorical', subset='training', seed=SEED)
 
rn_val_gen = ImageDataGenerator(
    preprocessing_function=resnet_preprocess, validation_split=0.2
).flow_from_directory(TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
                      class_mode='categorical', subset='validation', seed=SEED, shuffle=False)
 
# Test generators (Test.csv layout: Path, ClassId)
test_df = pd.read_csv(TEST_CSV).assign(ClassId=lambda d: d['ClassId'].astype(str))
 
mn_test_gen = ImageDataGenerator(preprocessing_function=mobilenet_preprocess).flow_from_dataframe(
    test_df, directory=os.path.dirname(TEST_CSV), x_col='Path', y_col='ClassId',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)
rn_test_gen = ImageDataGenerator(preprocessing_function=resnet_preprocess).flow_from_dataframe(
    test_df, directory=os.path.dirname(TEST_CSV), x_col='Path', y_col='ClassId',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)
plain_test_gen = ImageDataGenerator(rescale=1./255).flow_from_dataframe(
    test_df, directory=os.path.dirname(TEST_CSV), x_col='Path', y_col='ClassId',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)
 

In [ ]:
def build_transfer_model(base_fn, model_name):
    """
    Two-stage fine-tuning: Phase 1 = frozen base, train classification head.
    Architecture: Base → GlobalAvgPool → Dense(256, ReLU) → BN →
                  Dropout(0.4) → Dense(128, ReLU) → Dropout(0.3) → Dense(43, softmax)
    """
    base = base_fn(include_top=False, weights='imagenet', input_shape=(*IMG_SIZE, 3))
    base.trainable = False    # Phase 1: freeze base
 
    model = models.Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(NUM_CLASSES, activation='softmax'),
    ], name=model_name)
 
    model.compile(optimizer=Adam(1e-3),
                  loss='categorical_crossentropy', metrics=['accuracy'])
    return model, base

In [ ]:

print("  TRANSFER LEARNING — MobileNetV2")

mobilenet_model, mb_base = build_transfer_model(MobileNetV2, 'MobileNetV2_ADAS')
mobilenet_model.summary()
 
t0 = time.time()
mb_h = mobilenet_model.fit(
    mn_train_gen, validation_data=mn_val_gen, epochs=EPOCHS_TL,
    class_weight=class_weight,
    callbacks=[
        callbacks.EarlyStopping(monitor='val_loss', patience=4,
                                restore_best_weights=True, verbose=1),
        callbacks.ModelCheckpoint('mobilenet_best.keras', monitor='val_accuracy',
                                  save_best_only=True, verbose=0),
    ],
    verbose=1
)
mb_train_min = (time.time() - t0) / 60
mobilenet_model.save('mobilenet_best.keras')
print(f"\n MobileNetV2 training complete  —  {mb_train_min:.1f} min")
print(f"   Best Val Accuracy : {max(mb_h.history['val_accuracy'])*100:.2f}%")

In [ ]:

print("  TRANSFER LEARNING — ResNet50")

resnet_model, rn_base = build_transfer_model(ResNet50, 'ResNet50_ADAS')
resnet_model.summary()
 
t0 = time.time()
rn_h = resnet_model.fit(
    rn_train_gen, validation_data=rn_val_gen, epochs=EPOCHS_TL,
    class_weight=class_weight,
    callbacks=[
        callbacks.EarlyStopping(monitor='val_loss', patience=4,
                                restore_best_weights=True, verbose=1),
        callbacks.ModelCheckpoint('resnet_best.keras', monitor='val_accuracy',
                                  save_best_only=True, verbose=0),
    ],
    verbose=1
)
rn_train_min = (time.time() - t0) / 60
resnet_model.save('resnet_best.keras')
print(f"\n ResNet50 training complete  —  {rn_train_min:.1f} min")
print(f"   Best Val Accuracy : {max(rn_h.history['val_accuracy'])*100:.2f}%")
 

In [ ]:
def measure_inference_ms(model, generator, n=100):
    """
    Consistent methodology (as per project spec):
    Time model.predict() on exactly 100 images and divide by 100 → ms/image.
    Includes one warm-up pass to avoid first-call overhead on GPU/CPU.
    """
    generator.reset()
    batches = []
    for bx, _ in generator:
        batches.append(bx)
        if sum(len(b) for b in batches) >= n:
            break
    sample = np.vstack(batches)[:n]
    _  = model.predict(sample[:10], verbose=0)          # warm-up
    t0 = time.time()
    _  = model.predict(sample, verbose=0, batch_size=n)
    return round((time.time() - t0) * 1000 / n, 3)
 
def get_size_mb(path):
    if os.path.exists(path):
        return round(os.path.getsize(path) / (1024 ** 2), 2)
    return "N/A"
 

In [ ]:
custom_cnn   = tf.keras.models.load_model(best_cnn_path)
mobilenet_m  = tf.keras.models.load_model('mobilenet_best.keras')
resnet_m     = tf.keras.models.load_model('resnet_best.keras')
 
print("\nEvaluating models on held-out test set…")
cnn_loss, cnn_acc = custom_cnn.evaluate(plain_test_gen,  verbose=0)
mb_loss,  mb_acc  = mobilenet_m.evaluate(mn_test_gen,    verbose=0)
rn_loss,  rn_acc  = resnet_m.evaluate(rn_test_gen,       verbose=0)
 
print("Measuring inference times (100 images each)…")
cnn_ms = measure_inference_ms(custom_cnn,  plain_test_gen)
mb_ms  = measure_inference_ms(mobilenet_m, mn_test_gen)
rn_ms  = measure_inference_ms(resnet_m,    rn_test_gen)
 
tradeoff_df = pd.DataFrame([
    {
        "Model":                    "Custom CNN",
        "Val Accuracy (%)":         round(best_cnn_val * 100, 2),
        "Test Accuracy (%)":        round(cnn_acc * 100, 2),
        "Inference Time (ms/img)":  cnn_ms,
        "Model File Size (MB)":     get_size_mb(best_cnn_path),
        "Training Time (min)":      "~varies",
    },
    {
        "Model":                    "MobileNetV2",
        "Val Accuracy (%)":         round(max(mb_h.history['val_accuracy']) * 100, 2),
        "Test Accuracy (%)":        round(mb_acc * 100, 2),
        "Inference Time (ms/img)":  mb_ms,
        "Model File Size (MB)":     get_size_mb('mobilenet_best.keras'),
        "Training Time (min)":      round(mb_train_min, 1),
    },
    {
        "Model":                    "ResNet50",
        "Val Accuracy (%)":         round(max(rn_h.history['val_accuracy']) * 100, 2),
        "Test Accuracy (%)":        round(rn_acc * 100, 2),
        "Inference Time (ms/img)":  rn_ms,
        "Model File Size (MB)":     get_size_mb('resnet_best.keras'),
        "Training Time (min)":      round(rn_train_min, 1),
    },
])
 
print("\n" + "="*75)
print("  SPEED vs. ACCURACY TRADE-OFF TABLE")
print("="*75)
print(tradeoff_df.to_string(index=False))
 

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))
x     = np.arange(3)
cols  = ['#2196F3', '#4CAF50', '#FF5722']
acc   = tradeoff_df['Test Accuracy (%)'].astype(float)
infer = tradeoff_df['Inference Time (ms/img)'].astype(float)
 
bars = ax1.bar(x, acc, 0.45, color=cols, alpha=0.85, label='Test Accuracy (%)')
for b, v in zip(bars, acc):
    ax1.text(b.get_x() + b.get_width()/2, v + 0.3, f"{v:.1f}%",
             ha='center', fontsize=11, fontweight='bold')
ax1.set_ylabel("Test Accuracy (%)", fontsize=12, color='navy')
ax1.set_ylim(0, 115)
ax1.tick_params(axis='y', labelcolor='navy')
 
ax2 = ax1.twinx()
ax2.plot(x, infer, 'D--', color='crimson', lw=2.5, markersize=10,
         label='Inference Time (ms/img)')
for xi, yi in zip(x, infer):
    ax2.annotate(f"{yi} ms", (xi, yi), textcoords="offset points",
                 xytext=(10, 5), fontsize=10, color='crimson', fontweight='bold')
ax2.set_ylabel("Inference Time (ms / image)", fontsize=12, color='crimson')
ax2.tick_params(axis='y', labelcolor='crimson')
 
ax1.set_xticks(x)
ax1.set_xticklabels(tradeoff_df['Model'], fontsize=12)
ax1.set_title("ADAS Speed vs. Accuracy Trade-off Analysis",
              fontsize=14, fontweight='bold')
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc='upper right', fontsize=10)
ax1.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig("tradeoff_dual_axis_chart.png", dpi=150)
plt.show()
 

In [ ]:
mb_row = tradeoff_df.set_index('Model').loc['MobileNetV2']
rn_row = tradeoff_df.set_index('Model').loc['ResNet50']
 
latency_diff = float(rn_row['Inference Time (ms/img)']) - float(mb_row['Inference Time (ms/img)'])

In [ ]:
FINAL_MODEL    = mobilenet_m
FINAL_TEST_GEN = mn_test_gen
 
print("\n" + "="*55)
print("  SECTION 8 — FINAL MODEL EVALUATION (MobileNetV2)")
print("="*55)
 
FINAL_TEST_GEN.reset()
test_loss, test_acc = FINAL_MODEL.evaluate(FINAL_TEST_GEN, verbose=1)
print(f"\nFinal Test Accuracy : {test_acc*100:.2f}%   |   Loss : {test_loss:.4f}")

In [ ]:
FINAL_TEST_GEN.reset()
y_pred_prob = FINAL_MODEL.predict(FINAL_TEST_GEN, verbose=1)
y_pred      = np.argmax(y_pred_prob, axis=1)
y_true      = FINAL_TEST_GEN.classes[:len(y_pred)]

In [ ]:
print("\n Classification Report (Precision / Recall / F1 per class):")
report = classification_report(
    y_true, y_pred,
    target_names=[CLASS_NAMES[i] for i in range(NUM_CLASSES)],
    digits=3
)
print(report)

In [ ]:
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(22, 20))
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues', ax=ax,
            xticklabels=range(NUM_CLASSES),
            yticklabels=range(NUM_CLASSES),
            linewidths=0.4, linecolor='lightgrey')
ax.set_xlabel("Predicted Class", fontsize=13, fontweight='bold')
ax.set_ylabel("True Class",      fontsize=13, fontweight='bold')
ax.set_title("43×43 Confusion Matrix — Final ADAS Model (MobileNetV2)",
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig("confusion_matrix_43x43.png", dpi=150)
plt.show()


In [ ]:
cm_no_diag = cm.copy()
np.fill_diagonal(cm_no_diag, 0)
r, c = np.unravel_index(cm_no_diag.argmax(), cm_no_diag.shape)
print(f"\n  Most confused pair:")
print(f"   True  [{r:2d}]: {CLASS_NAMES[r]}")
print(f"   Pred  [{c:2d}]: {CLASS_NAMES[c]}  →  {cm_no_diag[r, c]} misclassifications")
print("   Likely cause: both are red circular signs with similar numerals / shapes.")
 

In [ ]:
FINAL_TEST_GEN.reset()
bx, by = next(FINAL_TEST_GEN)
preds  = np.argmax(FINAL_MODEL.predict(bx[:9], verbose=0), axis=1)
trues  = np.argmax(by[:9], axis=1)
 
fig, axes = plt.subplots(3, 3, figsize=(13, 13))
for i, ax in enumerate(axes.flat):
    # Reverse mobilenet preprocessing for display
    disp = bx[i].copy()
    disp = (disp - disp.min()) / (disp.max() - disp.min() + 1e-8)
    ax.imshow(np.clip(disp, 0, 1))
    correct = trues[i] == preds[i]
    color   = 'green' if correct else 'red'
    ax.set_title(
        f"True : {CLASS_NAMES[trues[i]][:22]}\nPred : {CLASS_NAMES[preds[i]][:22]}",
        color=color, fontsize=8, fontweight='bold'
    )
    for spine in ax.spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(4)
    ax.axis('off')
plt.suptitle("Test Set Predictions  (Green = Correct  |  Red = Incorrect)",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("sample_predictions_grid.png", dpi=150)
plt.show()

In [ ]:
def run_full_pipeline(image_input, model, class_names=CLASS_NAMES):
    """
    Complete two-stage ADAS pipeline on a single image.
 
    Stage 1 — Detection  : detect_and_crop_sign() → bounding box + cropped region
    Stage 2 — Classification : MobileNetV2 → predicted label + confidence + top-3
 
    Returns
    -------
    dict with keys: original_rgb, annotated_rgb, cropped_64x64, prediction
    """
    if isinstance(image_input, str):
        img_bgr = cv2.imread(image_input)
    else:
        img_bgr = image_input.copy()
 
    result = detect_and_crop_sign(img_bgr)
 
    if result is None:
        return None
 
    img_bgr, (bx, by, bw, bh), cropped = result
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
 
    # Draw bounding box
    annotated = img_rgb.copy()
    cv2.rectangle(annotated, (bx, by), (bx + bw, by + bh), (0, 255, 0), 3)
 
    # Classify
    inp   = mobilenet_preprocess(cropped.astype(np.float32))
    probs = model.predict(np.expand_dims(inp, 0), verbose=0)[0]
    top3  = probs.argsort()[-3:][::-1]
 
    prediction = {
        "class_id":   int(top3[0]),
        "class_name": class_names[top3[0]],
        "confidence": float(probs[top3[0]]),
        "top3":       [(class_names[i], float(probs[i])) for i in top3],
    }
    cv2.putText(annotated,
                f"{prediction['class_name'][:22]} {prediction['confidence']*100:.0f}%",
                (bx, max(by - 8, 12)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 0), 2)
 
    return {
        "original_rgb":  img_rgb,
        "annotated_rgb": annotated,
        "cropped":       cropped,
        "prediction":    prediction,
    }
 
 

In [ ]:
e2e_classes = [14, 17]
 
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("End-to-End ADAS Pipeline: Scene → Detection → Classification",
             fontsize=14, fontweight='bold')
 
for row, cls_id in enumerate(e2e_classes):
    scene_bgr, _ = create_synthetic_scene(cls_id)
    out          = run_full_pipeline(scene_bgr, mobilenet_m)
 
    axes[row, 0].imshow(cv2.cvtColor(scene_bgr, cv2.COLOR_BGR2RGB))
    axes[row, 0].set_title("Stage 0: Full Road Scene", fontsize=10)
 
    if out is not None:
        axes[row, 1].imshow(out['annotated_rgb'])
        axes[row, 1].set_title("Stage 1: Detection + Bounding Box", fontsize=10)
 
        axes[row, 2].imshow(out['cropped'])
        pred    = out['prediction']
        correct = pred['class_id'] == cls_id
        color   = 'green' if correct else 'red'
        axes[row, 2].set_title(
            f"Stage 2: Classification\n"
            f"{'✅' if correct else '❌'} {pred['class_name'][:24]}\n"
            f"Confidence: {pred['confidence']*100:.1f}%",
            fontsize=9, color=color
        )
 
        print(f"Test {row+1}: True = {CLASS_NAMES[cls_id]}")
        print(f"         Predicted = {pred['class_name']} ({pred['confidence']*100:.1f}%)")
        print(f"         Top-3 = {pred['top3']}\n")
    else:
        axes[row, 1].text(0.5, 0.5, "Detection Failed", ha='center', va='center',
                          color='red', transform=axes[row, 1].transAxes)
        axes[row, 2].axis('off')
 
    for col in range(3):
        axes[row, col].axis('off')
 
plt.tight_layout()
plt.savefig("e2e_pipeline_demo.png", dpi=150)
plt.show()